In [ ]:
import sys
from awsglue.dynamicframe import DynamicFrame
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
import pyspark.sql.functions as sf
from pyspark.sql.functions import to_timestamp, when, col, year, month, dayofmonth, hour

In [ ]:
#Job parameters
args = getResolvedOptions(sys.argv, ["JOB_NAME","INPUT_LOCATION","OUTPUT_LOCATION"])

In [ ]:
#set input and output paths
input_path = args["INPUT_LOCATION"]
output_path = args["OUTPUT_LOCATION"]

In [ ]:
#Initialize Glue context
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args["JOB_NAME"], args)

In [ ]:
# Read with DynamicFrame (required for bookmarks)
dynamic_frame = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [input_path], "recurse": True},
    format="json",
    transformation_ctx="read_raw_event_logs"  # REQUIRED: Unique identifier for bookmark tracking
)

In [ ]:

# Convert to DataFrame for transformations
df = dynamic_frame.toDF()

In [ ]:
#convert data type for timestamp
df = df.withColumn("ts",to_timestamp(col("timestamp")))

In [ ]:

#partition derivation using year, month, dayofmonth, hour functions
df = df.withColumn("year",year(col("ts")).cast("string"))
df = df.withColumn("month",month(col("ts")).cast("string"))
df = df.withColumn("day",dayofmonth(col("ts")).cast("string"))
df = df.withColumn("hour",hour(col("ts")).cast("string"))


In [ ]:
#final df with ts column dropped. 
final_df = df.drop("ts")

In [ ]:
# Convert back to DynamicFrame before writing
output_dyf = DynamicFrame.fromDF(final_df, glueContext, "output")

In [ ]:
# Write using Glue writer (preserves bookmark state)
glueContext.write_dynamic_frame.from_options(
    frame=output_dyf,
    connection_type="s3",
    connection_options={"path": output_path, "partitionKeys": ["year", "month", "day", "hour"]},
    format="parquet",
    format_options={"compression":"snappy"},
    transformation_ctx="write_parquet_logs"  # REQUIRED: Unique identifier for bookmark tracking
)

In [ ]:
job.commit()
